# Layer-wise Hidden State Extraction for Codenames Duet — Random Qwen2.5-7B

**Variant:** Randomly-initialized weights, trained-Qwen architecture.

**Companion to:** `Qwen_Codenames_Layer_Extraction.ipynb` (trained weights).

This notebook is the random-init control for Qwen2.5-7B-Instruct. The
architecture, tokenizer, prompts, dataset, sampling, pooling, metrics, and
output schema are byte-identical to the trained-Qwen notebook. The single
substantive difference is that model weights are drawn from the model's own
initialization distribution (normal, σ = `config.initializer_range`,
typically 0.02) rather than loaded from the pretrained checkpoint. This
isolates the contribution of the trained-Qwen *architecture* (causal masking
+ RoPE + GQA + 7B parameter scale) from the contribution of *training*.

Generation is disabled because a randomly-initialized causal LM produces
degenerate output (the LM head is also random). This matches the structure
of the encoder-only notebooks (BERT, T5, ModernBERT) which also do not
generate.


## Protocol

**What is controlled:**

- Identical model architecture: `Qwen/Qwen2.5-7B-Instruct` config loaded
  via `AutoConfig.from_pretrained`, instantiated via
  `AutoModelForCausalLM.from_config`. All hyperparameters (depth, hidden
  dim, head count, RoPE theta, intermediate size, RMSNorm, SwiGLU) are
  inherited from the pretrained config.
- Identical tokenizer: `AutoTokenizer.from_pretrained` against the same
  pretrained checkpoint. Tokenization is independent of weights, so the
  prompt tokenization is byte-identical to the trained run.
- Identical experimental contract (`CONTRACT_v1.0`): SAMPLE_SIZE=2000,
  RANDOM_SEED=42, two conditions (no_social / with_social), two pooling
  methods (mean / max_norm), N_SHUFFLES=2, alphabetical candidate order,
  N=100 vector subsample, 200-board shards.
- Identical dataset and sampling: `clue_generation.csv`, seed 42 sampling,
  same row_ids as every other model in the 6-model suite.

**What changes:**

- Model weights: random, drawn from `config.initializer_range`-scaled
  normal distribution per Qwen's `_init_weights` method. This corresponds
  to the state of Qwen2.5 at training step 0.
- Generation: disabled. The `generate_response` utility is omitted, which
  causes the main loop's `HAS_GENERATION` flag to evaluate False. This
  matches BERT/T5/ModernBERT.
- File prefix: `random_qwen` (vs `qwen`). Outputs do not overwrite trained
  Qwen outputs.

**What stays the same in the output schema:**

Per condition (no_social, with_social), the notebook produces:

- `random_qwen_general_{mode}.csv` — one row per (board × permutation),
  with predicted words, accuracy flags, MRR, Hit@k, rank stats, margins.
- `random_qwen_metrics_{mode}.parquet` — one row per
  (board × permutation × layer × word × pooling), with cosine_to_hint,
  rank, anisotropy.
- `random_qwen_vectors_subsample_index_{mode}.csv` and
  `random_qwen_vectors_subsample_{mode}_f16.npz` — N=100 raw vectors.
- `random_qwen_layer_margins_{pm}_{mode}.csv` — per-layer margin curves.
- `random_qwen_position_confound_by_layer.csv` — per-layer Spearman ρ.
- `random_qwen_shuffle_decomposition_by_layer.csv` — variance decomposition.

No `random_qwen_generation_{mode}.csv` is produced.

**Pre-flight check (Cell 5):** Before running the full N=2000 loop, a small
diagnostic forward pass is executed on 5 boards to verify the random model
produces numerically stable hidden states. Three checks: (a) hidden state
norms across layers, (b) NaN/Inf count, (c) anisotropy at layer 0. If any
hard threshold fails, the loop should not be run; investigate fp16
stability and consider bf16 fallback.

**Random seed handling for weight init:** `torch.manual_seed(RANDOM_SEED)`
is set in Cell 2 (global config) and re-set immediately before
`from_config` in Cell 4, defensively. Any change to RANDOM_SEED will
produce a different random-init network, but the dataset sampling is also
seeded with the same value, so a single seed change re-randomizes both
weights and sample. For independent control of these, modify Cell 4
locally.


## Cell 1 — Global Configuration


In [ ]:
import gc
import os
import ast
import re
import random
import numpy as np
import torch
import pandas as pd
import torch.nn.functional as F
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from scipy.stats import spearmanr, mannwhitneyu, norm as scipy_norm

gc.collect()
torch.cuda.empty_cache()

# --- Reproducibility ---
# RANDOM_SEED governs three things simultaneously:
#   1. Python / numpy / torch global RNG state
#   2. Dataset sampling (Cell 9)
#   3. Random weight initialization (Cell 4, re-seeded defensively before init)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# === MODEL-SPECIFIC: model identity =========================================
# MODEL_NAME points to the same HuggingFace repo as the trained-Qwen notebook;
# we use it ONLY to fetch the architecture config and the tokenizer. The
# weights from this checkpoint are NOT loaded — see Cell 4.
MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"
MODEL_PREFIX = "random_qwen"
BASE_DIR     = "/content/drive/MyDrive/TCC/random_qwen_outputs"
# ============================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Experiment (frozen by CONTRACT_v1.0 Section 1) ---
EXPERIMENT_MODES      = [False, True]        # False = no_social, True = with_social
SAMPLE_SIZE           = 2000
CANDIDATE_ORDER       = "fixed"               # alphabetical within board
POOLING_METHODS       = ["mean", "max_norm"]  # both run in parallel
VECTOR_SUBSAMPLE_SIZE = 100
N_SHUFFLES            = 2                     # canonical + 2 random permutations
GENERATION_MAX_TOKENS = 30                    # unused here (generation disabled)
SHARD_BOARDS          = 200                   # flush Stream A every K boards

print("=" * 60)
print("GLOBAL CONFIGURATION (Contract v1.0) — RANDOM-INIT VARIANT")
print("=" * 60)
print(f"MODEL_NAME              : {MODEL_NAME}")
print(f"MODEL_PREFIX            : {MODEL_PREFIX}")
print(f"BASE_DIR                : {BASE_DIR}")
print(f"DEVICE                  : {DEVICE}")
print(f"RANDOM_SEED             : {RANDOM_SEED}")
print(f"SAMPLE_SIZE             : {SAMPLE_SIZE}")
print(f"CANDIDATE_ORDER         : {CANDIDATE_ORDER}")
print(f"POOLING_METHODS         : {POOLING_METHODS}")
print(f"VECTOR_SUBSAMPLE_SIZE   : {VECTOR_SUBSAMPLE_SIZE}")
print(f"N_SHUFFLES              : {N_SHUFFLES}")
print(f"SHARD_BOARDS            : {SHARD_BOARDS}")
print(f"MODES                   : {['no_social', 'with_social']}")
print()
print("Weight source           : RANDOM INIT (config.initializer_range, normal)")
print("Generation              : DISABLED (random LM head produces noise)")


GLOBAL CONFIGURATION (Contract v1.0) — RANDOM-INIT VARIANT
MODEL_NAME              : Qwen/Qwen2.5-7B-Instruct
MODEL_PREFIX            : random_qwen
BASE_DIR                : /content/drive/MyDrive/TCC/random_qwen_outputs
DEVICE                  : cuda
RANDOM_SEED             : 42
SAMPLE_SIZE             : 2000
CANDIDATE_ORDER         : fixed
POOLING_METHODS         : ['mean', 'max_norm']
VECTOR_SUBSAMPLE_SIZE   : 100
N_SHUFFLES              : 2
SHARD_BOARDS            : 200
MODES                   : ['no_social', 'with_social']

Weight source           : RANDOM INIT (config.initializer_range, normal)
Generation              : DISABLED (random LM head produces noise)


## Cell 2 — Load Tokenizer and Instantiate Random-Init Model

The tokenizer is loaded from the pretrained checkpoint (tokenization is
weight-independent). The model is instantiated from the config alone via
`AutoModelForCausalLM.from_config`, which:

1. Constructs the `Qwen2ForCausalLM` module hierarchy with the exact
   architecture of Qwen2.5-7B-Instruct.
2. Initializes every parameter through the model's `_init_weights` method,
   which draws from `N(0, config.initializer_range²)` for Linear and
   Embedding layers, sets LayerNorm/RMSNorm weights to 1.0, and leaves
   biases at 0.

We re-seed `torch.manual_seed(RANDOM_SEED)` immediately before this call
so that weight initialization is deterministic regardless of any earlier
RNG consumption. The result is reproducible: re-running this cell with
the same seed produces a parameter-identical model.

`dtype=torch.float16` matches the trained-Qwen notebook for direct
comparison. If the pre-flight diagnostic (Cell 5) flags fp16 instability,
re-instantiate with `dtype=torch.bfloat16` and document the fallback in
the Methods chapter.


In [ ]:
# === MODEL-SPECIFIC: random-init model construction =========================
print(f"\nLoading tokenizer from: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Causal LM requirement preserved from the trained notebook. The pad_token
# is a tokenizer-level concern, not a weight-level one.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Fetching architecture config from: {MODEL_NAME}")
config = AutoConfig.from_pretrained(MODEL_NAME)

# Defensive re-seed: ensures the weight init below is deterministic even if
# any prior cell consumed torch RNG state.
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Instantiating model from config with random weights (seed={RANDOM_SEED})...")
print(f"  initializer_range from config: {getattr(config, 'initializer_range', 'N/A')}")

# from_config builds the architecture and runs _init_weights. No pretrained
# weights are loaded.
model = AutoModelForCausalLM.from_config(
    config,
    torch_dtype=torch.float16,
)
model = model.to(DEVICE)
model.eval()
# ============================================================================

NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size

print(f"\nModel instantiated successfully.")
print(f"  Number of transformer layers : {NUM_LAYERS}")
print(f"  Hidden state dimensionality  : {HIDDEN_DIM}")
print(f"  Total hidden states per token: {NUM_LAYERS + 1}  (embedding + {NUM_LAYERS} layers)")
print(f"  Vocabulary size              : {model.config.vocab_size}")
print(f"  dtype                        : {next(model.parameters()).dtype}")
print(f"  Device                       : {next(model.parameters()).device}")

# --- Quick parameter sanity: weights should NOT match the pretrained checkpoint
# We do not load pretrained weights; if they were loaded by mistake, the first
# Linear weight's standard deviation would be much smaller than initializer_range.
_first_linear = None
for _m in model.modules():
    if isinstance(_m, torch.nn.Linear):
        _first_linear = _m
        break
if _first_linear is not None:
    _w = _first_linear.weight.detach().float()
    print(f"\n  Sanity (first Linear weight): mean={_w.mean().item():+.6f}, "
          f"std={_w.std().item():.6f}")
    print(f"  Expected std ≈ {config.initializer_range} (random init)")



Loading tokenizer from: Qwen/Qwen2.5-7B-Instruct
Fetching architecture config from: Qwen/Qwen2.5-7B-Instruct
Instantiating model from config with random weights (seed=42)...
  initializer_range from config: 0.02

Model instantiated successfully.
  Number of transformer layers : 28
  Hidden state dimensionality  : 3584
  Total hidden states per token: 29  (embedding + 28 layers)
  Vocabulary size              : 152064
  dtype                        : torch.float16
  Device                       : cuda:0

  Sanity (first Linear weight): mean=-0.000002, std=0.019981
  Expected std ≈ 0.02 (random init)


## Cell 3 — Load and Prepare Dataset


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"

df = pd.read_csv(DATASET_PATH)

for col in ["targets", "black", "tan"]:
    df[col] = df[col].apply(ast.literal_eval)

def build_candidates_fixed_order(row) -> List[str]:
    """Returns all board words in stable alphabetical order."""
    all_words = list(row["targets"]) + list(row["black"]) + list(row["tan"])
    return sorted(all_words)

df["candidates"] = df.apply(build_candidates_fixed_order, axis=1)
df = df.reset_index(drop=True)
df["row_id"] = df.index.astype(int)

print(f"Dataset loaded. Shape: {df.shape}")
print(f"Candidate ordering: {CANDIDATE_ORDER} (alphabetical)")
print(f"\nExample board (row 0):")
print(f"  hint       : {df.loc[0, 'output']}")
print(f"  targets    : {df.loc[0, 'targets']}")
print(f"  candidates : {df.loc[0, 'candidates'][:5]} ...")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded. Shape: (7703, 48)
Candidate ordering: fixed (alphabetical)

Example board (row 0):
  hint       : sandwich
  targets    : ['sub']
  candidates : ['beat', 'boom', 'charge', 'check', 'drop'] ...


## Cell 4 — Define Giver Feature Columns


In [ ]:
GIVER_COLS = [
    "giver.marriage",
    "giver.education",
    "giver.race",
    "giver.continent",
    "giver.language",
    "giver.religion",
    "giver.gender",
    "giver.country",
    "giver.political",
]

missing = [c for c in GIVER_COLS if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing giver columns: {missing}\n"
        f"Available: {list(df.columns)}"
    )

print(f"All {len(GIVER_COLS)} giver feature columns validated.")


All 9 giver feature columns validated.


## Cell 5 — Prompt Builder

Identical to the trained-Qwen notebook. The chat template wrapping
(`apply_chat_template` with ChatML delimiters) is preserved because the
tokenization must remain identical to the trained run for the comparison
to be clean. A random-init model still consumes the same input tokens;
what differs is what those tokens map to in the (random) embedding space.


In [ ]:
def _format_feature_value(v) -> str:
    """Formats a feature value as a clean string. Returns 'NA' for missing."""
    if pd.isna(v):
        return "NA"
    if isinstance(v, float):
        return f"{v:.4f}".rstrip("0").rstrip(".")
    return str(v)

_FEATURE_LABEL_MAP = {
    "giver.marriage"  : "Marriage",
    "giver.education" : "Education",
    "giver.race"      : "Race",
    "giver.continent" : "Continent",
    "giver.language"  : "Language",
    "giver.religion"  : "Religion",
    "giver.gender"    : "Gender",
    "giver.country"   : "Country",
    "giver.political" : "Politics",
    "giver.politics"  : "Politics",  # T5 dataset variant
}


def build_instruction_body(
    hint: str,
    candidates: List[str],
    giver_features: Optional[Dict[str, object]],
    use_social_context: bool,
) -> Tuple[str, Dict[str, str]]:
    """
    Constructs the SHARED instruction body (Format A from CONTRACT_v1.0 Section 2).
    Byte-identical across all six notebooks.
    """
    words_block = "\n".join(
        [f"{i+1}. {w}" for i, w in enumerate(candidates)]
    )
    feature_markers = {}

    if use_social_context and giver_features:
        parts = []
        for k, v in giver_features.items():
            v_str = _format_feature_value(v)
            label = _FEATURE_LABEL_MAP.get(k, k.split(".")[-1].capitalize())
            marker = f"{label}: {v_str}"
            feature_markers[k] = marker
            parts.append(marker)
        social_block = ", ".join(parts) if parts else "N/A"

        instruction_body = (
            f"You are playing the game Codenames.\n"
            f"The clue was given by a player with the following characteristics:\n"
            f"{social_block}\n"
            f'The hint is: "{hint}"\n'
            f"The possible words are:\n"
            f"{words_block}\n"
            f"Which word best matches the hint? Only output the word."
        )
    else:
        instruction_body = (
            f"You are playing the game Codenames.\n"
            f'The hint is: "{hint}"\n'
            f"The possible words are:\n"
            f"{words_block}\n"
            f"Which word best matches the hint? Only output the word."
        )

    return instruction_body, feature_markers


def build_prompt(
    hint: str,
    candidates: List[str],
    giver_features: Optional[Dict[str, object]],
    use_social_context: bool,
) -> Tuple[str, Dict[str, str]]:
    """
    Returns (prompt, feature_markers) where prompt is wrapped in Qwen's
    native ChatML chat template via tokenizer.apply_chat_template().
    """
    instruction_body, feature_markers = build_instruction_body(
        hint=hint,
        candidates=candidates,
        giver_features=giver_features,
        use_social_context=use_social_context,
    )

    # === MODEL-SPECIFIC: chat template wrapping =============================
    # Qwen2.5 uses ChatML with <|im_start|>/<|im_end|> delimiters.
    # We use apply_chat_template to let the tokenizer construct the correct
    # format. add_generation_prompt=True appends the assistant-turn header.
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": instruction_body},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    # ========================================================================

    return prompt, feature_markers


## Cell 6 — Token Span Detection and Pooling Utilities


In [ ]:
def find_token_spans(
    full_text: str,
    offset_mapping: List[Tuple[int, int]],
    spans_to_find: Dict[str, str],
    candidate_anchor: str = "The possible words are:",
) -> Dict[str, Tuple[int, int]]:
    """
    Locates token-level spans for target substrings in the prompt.

    Parameters
    ----------
    full_text : str
        The full prompt string.
    offset_mapping : list of (int, int)
        Character-level offsets for each token.
    spans_to_find : dict
        Maps span name -> substring. "cand:" prefix searches after anchor.
    candidate_anchor : str
        Delimiter marking the start of the candidate list.

    Returns
    -------
    found : dict
        Maps span name -> (token_start, token_end) index tuple.
    """
    found = {}
    cand_start_char = full_text.find(candidate_anchor)
    if cand_start_char == -1:
        cand_start_char = 0

    for name, substring in spans_to_find.items():
        if name.startswith("cand:"):
            char_start = full_text.find(substring, cand_start_char)
        else:
            char_start = full_text.find(substring)

        if char_start == -1:
            continue

        char_end = char_start + len(substring)

        token_start, token_end = None, None
        for idx, (s, e) in enumerate(offset_mapping):
            if s < char_end and e > char_start:
                if token_start is None:
                    token_start = idx
                token_end = idx + 1

        if token_start is not None and token_end > token_start:
            found[name] = (token_start, token_end)

    return found


def mean_pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
) -> Optional[np.ndarray]:
    """Mean-pools hidden states across a token span. Returns float16."""
    s, e = span
    tokens = layer_hidden_states[s:e]
    if tokens.shape[0] == 0:
        return None
    pooled = tokens.mean(dim=0).detach().float().cpu().numpy()
    return pooled.astype(np.float16)


def max_norm_pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
) -> Optional[np.ndarray]:
    """
    Selects the token with the highest L2 norm from a span. Returns float16.
    Avoids creating synthetic composite vectors. The highest-norm token
    typically corresponds to the root morpheme.
    """
    s, e = span
    tokens = layer_hidden_states[s:e]
    if tokens.shape[0] == 0:
        return None
    if tokens.shape[0] == 1:
        return tokens[0].detach().float().cpu().numpy().astype(np.float16)
    norms = torch.norm(tokens, p=2, dim=1)
    best_idx = torch.argmax(norms).item()
    pooled = tokens[best_idx].detach().float().cpu().numpy()
    return pooled.astype(np.float16)


def pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
    method: str = "mean",
) -> Optional[np.ndarray]:
    """Dispatcher for pooling methods. Supports 'mean' and 'max_norm'."""
    if method == "mean":
        return mean_pool_span(layer_hidden_states, span)
    elif method == "max_norm":
        return max_norm_pool_span(layer_hidden_states, span)
    else:
        raise ValueError(f"Unknown pooling method: {method}")


def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1D arrays. Returns 0.0 on zero norm."""
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


## Cell 7 — Core Instance Processing Function


In [ ]:
def extract_giver_features(
    row: pd.Series,
    giver_cols: List[str],
) -> Dict[str, object]:
    """Extracts non-null giver feature values from a dataset row."""
    return {
        c: row[c]
        for c in giver_cols
        if c in row.index and not pd.isna(row[c])
    }


def run_instance(
    row: pd.Series,
    giver_cols: List[str],
    use_social_context: bool,
    candidates_order: List[str],
    permutation_id: int = 0,
    save_vectors: bool = False,
) -> Tuple[Dict, List[Dict], Optional[List[Dict]]]:
    """
    Processes one board instance under a given candidate ordering.

    Parameters
    ----------
    row : pd.Series
        One row from the sampled dataset.
    giver_cols : list of str
        Column names for giver demographic features.
    use_social_context : bool
        If True, includes giver features in the prompt.
    candidates_order : list of str
        The candidate words in the desired order. For the canonical run
        this is alphabetical; for shuffles it is a random permutation.
    permutation_id : int
        0 = canonical (alphabetical) ordering. 1..K = shuffles.
    save_vectors : bool
        If True, raw vectors are retained for the subsample.
    """
    row_id     = int(row["row_id"])
    hint       = str(row["output"])
    candidates = list(candidates_order)
    targets    = set(row["targets"])
    black      = set(row["black"])
    tan        = set(row["tan"])

    giver_features = (
        extract_giver_features(row, giver_cols)
        if use_social_context else {}
    )

    prompt, feature_markers = build_prompt(
        hint=hint,
        candidates=candidates,
        giver_features=giver_features,
        use_social_context=use_social_context,
    )

    # --- Tokenization ---
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_offsets_mapping=True,
    ).to(DEVICE)

    offset_mapping     = inputs["offset_mapping"][0].tolist()
    prompt_token_count = inputs["input_ids"].shape[1]

    # --- Build span targets ---
    spans_to_find = {"hint": hint}
    for c in candidates:
        spans_to_find[f"cand:{c}"] = c
    if use_social_context:
        for k, marker in feature_markers.items():
            spans_to_find[f"giver:{k}"] = marker

    spans = find_token_spans(prompt, offset_mapping, spans_to_find)

    if "hint" not in spans:
        raise ValueError(
            f"Hint span not found for row_id={row_id}, hint='{hint}'."
        )

    candidate_position_map = {w: i for i, w in enumerate(candidates)}

    # === MODEL-SPECIFIC: forward pass with output_hidden_states ============
    # Causal LMs accept output_hidden_states at inference time.
    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
            return_dict=True,
        )
    # =======================================================================

    hidden_states = outputs.hidden_states

    # ================================================================
    # Compute metrics across ALL layers, for ALL pooling methods
    # ================================================================
    metrics_records = []
    vector_records  = [] if save_vectors else None

    for layer_idx in range(NUM_LAYERS + 1):
        layer_hs = hidden_states[layer_idx][0]

        # --- Pool hint vector ---
        hint_vecs = {pm: pool_span(layer_hs, spans["hint"], method=pm)
                     for pm in POOLING_METHODS}
        hint_token_count = spans["hint"][1] - spans["hint"][0]

        # --- Pool candidate vectors ---
        cand_vecs = {}
        cand_meta = {}
        for c in candidates:
            if c in targets:
                c_type = "target"
            elif c in black:
                c_type = "black"
            elif c in tan:
                c_type = "tan"
            else:
                c_type = "unknown"

            ck = f"cand:{c}"
            cand_vecs[c] = {}
            if ck in spans:
                c_token_count = spans[ck][1] - spans[ck][0]
                for pm in POOLING_METHODS:
                    cand_vecs[c][pm] = pool_span(layer_hs, spans[ck], method=pm)
            else:
                c_token_count = 0
                for pm in POOLING_METHODS:
                    cand_vecs[c][pm] = None
            cand_meta[c] = {"word_type": c_type, "token_count": c_token_count}

        # --- Pool giver feature vectors (with_social only) ---
        giver_vecs = {}
        giver_token_counts = {}
        if use_social_context:
            for feat_name, marker in feature_markers.items():
                fk = f"giver:{feat_name}"
                giver_vecs[feat_name] = {}
                if fk in spans:
                    giver_token_counts[feat_name] = spans[fk][1] - spans[fk][0]
                    for pm in POOLING_METHODS:
                        giver_vecs[feat_name][pm] = pool_span(layer_hs, spans[fk], method=pm)
                else:
                    giver_token_counts[feat_name] = 0
                    for pm in POOLING_METHODS:
                        giver_vecs[feat_name][pm] = None

        # --- Cosines: hint -> each candidate, per pooling method ---
        cosines_per_method = {}
        for pm in POOLING_METHODS:
            cosines_per_method[pm] = {}
            h_vec = hint_vecs[pm]
            if h_vec is None:
                for c in candidates:
                    cosines_per_method[pm][c] = float("nan")
                continue
            h_vec_f32 = h_vec.astype(np.float32)
            for c in candidates:
                c_vec = cand_vecs[c][pm]
                if c_vec is not None:
                    cosines_per_method[pm][c] = cosine_similarity_np(
                        h_vec_f32, c_vec.astype(np.float32)
                    )
                else:
                    cosines_per_method[pm][c] = float("nan")

        # --- Ranks per pooling method ---
        ranks_per_method = {}
        for pm in POOLING_METHODS:
            valid_cosines = {
                w: v for w, v in cosines_per_method[pm].items()
                if not np.isnan(v)
            }
            sorted_words = sorted(
                valid_cosines.keys(),
                key=lambda w: valid_cosines[w],
                reverse=True,
            )
            ranks_per_method[pm] = {}
            for rank_pos, w in enumerate(sorted_words, start=1):
                ranks_per_method[pm][w] = rank_pos
            for c in candidates:
                if c not in ranks_per_method[pm]:
                    ranks_per_method[pm][c] = float("nan")

        # --- All-pairs candidate cosines for anisotropy (mean pooling) ---
        all_pair_cosines_layer = []
        valid_cand_vecs_mean = []
        for c in candidates:
            v = cand_vecs[c]["mean"]
            if v is not None:
                valid_cand_vecs_mean.append(v.astype(np.float32))
        n_valid = len(valid_cand_vecs_mean)
        if n_valid >= 2:
            for i in range(n_valid):
                for j in range(i + 1, n_valid):
                    all_pair_cosines_layer.append(
                        cosine_similarity_np(
                            valid_cand_vecs_mean[i],
                            valid_cand_vecs_mean[j],
                        )
                    )

        if all_pair_cosines_layer:
            layer_aniso_mean = float(np.mean(all_pair_cosines_layer))
            layer_aniso_std  = float(np.std(all_pair_cosines_layer))
        else:
            layer_aniso_mean = float("nan")
            layer_aniso_std  = float("nan")

        # --- Build metric records: hint ---
        hint_metric = {
            "row_id"                     : row_id,
            "layer"                      : layer_idx,
            "word"                       : hint,
            "word_type"                  : "hint",
            "token_count"                : hint_token_count,
            "list_position"              : -1,
            "use_social_context"         : use_social_context,
            "permutation_id"             : permutation_id,
            "layer_mean_pairwise_cosine" : layer_aniso_mean,
            "layer_std_pairwise_cosine"  : layer_aniso_std,
        }
        for pm in POOLING_METHODS:
            hint_metric[f"cosine_to_hint_{pm}"]  = float("nan")
            hint_metric[f"rank_{pm}"]            = float("nan")
            hint_metric[f"reciprocal_rank_{pm}"] = float("nan")
        metrics_records.append(hint_metric)

        # --- Build metric records: candidates ---
        for c in candidates:
            c_metric = {
                "row_id"                     : row_id,
                "layer"                      : layer_idx,
                "word"                       : c,
                "word_type"                  : cand_meta[c]["word_type"],
                "token_count"                : cand_meta[c]["token_count"],
                "list_position"              : candidate_position_map[c],
                "use_social_context"         : use_social_context,
                "permutation_id"             : permutation_id,
                "layer_mean_pairwise_cosine" : layer_aniso_mean,
                "layer_std_pairwise_cosine"  : layer_aniso_std,
            }
            for pm in POOLING_METHODS:
                cos_val  = cosines_per_method[pm][c]
                rank_val = ranks_per_method[pm][c]
                c_metric[f"cosine_to_hint_{pm}"]  = cos_val
                c_metric[f"rank_{pm}"]            = rank_val
                c_metric[f"reciprocal_rank_{pm}"] = (
                    1.0 / rank_val if not np.isnan(rank_val) else float("nan")
                )
            metrics_records.append(c_metric)

        # --- Build metric records: giver features ---
        if use_social_context:
            for feat_name in feature_markers:
                gf_metric = {
                    "row_id"                     : row_id,
                    "layer"                      : layer_idx,
                    "word"                       : feat_name,
                    "word_type"                  : "giver_feature",
                    "token_count"                : giver_token_counts.get(feat_name, 0),
                    "list_position"              : -1,
                    "use_social_context"         : use_social_context,
                    "permutation_id"             : permutation_id,
                    "layer_mean_pairwise_cosine" : layer_aniso_mean,
                    "layer_std_pairwise_cosine"  : layer_aniso_std,
                }
                for pm in POOLING_METHODS:
                    h_vec = hint_vecs[pm]
                    g_vec = giver_vecs[feat_name].get(pm)
                    if h_vec is not None and g_vec is not None:
                        gf_metric[f"cosine_to_hint_{pm}"] = cosine_similarity_np(
                            h_vec.astype(np.float32), g_vec.astype(np.float32)
                        )
                    else:
                        gf_metric[f"cosine_to_hint_{pm}"] = float("nan")
                    gf_metric[f"rank_{pm}"]            = float("nan")
                    gf_metric[f"reciprocal_rank_{pm}"] = float("nan")
                metrics_records.append(gf_metric)

        # --- Save vectors (subsample, canonical only) ---
        if save_vectors:
            for pm in POOLING_METHODS:
                vector_records.append({
                    "row_id": row_id, "layer": layer_idx,
                    "word": hint, "word_type": "hint",
                    "token_count": hint_token_count,
                    "pooling_method": pm,
                    "use_social_context": use_social_context,
                    "vector": hint_vecs[pm],
                })
            for c in candidates:
                for pm in POOLING_METHODS:
                    vector_records.append({
                        "row_id": row_id, "layer": layer_idx,
                        "word": c, "word_type": cand_meta[c]["word_type"],
                        "token_count": cand_meta[c]["token_count"],
                        "pooling_method": pm,
                        "use_social_context": use_social_context,
                        "vector": cand_vecs[c][pm],
                    })
            if use_social_context:
                for feat_name in feature_markers:
                    for pm in POOLING_METHODS:
                        vector_records.append({
                            "row_id": row_id, "layer": layer_idx,
                            "word": feat_name, "word_type": "giver_feature",
                            "token_count": giver_token_counts.get(feat_name, 0),
                            "pooling_method": pm,
                            "use_social_context": use_social_context,
                            "vector": giver_vecs[feat_name].get(pm),
                        })

    # ================================================================
    # Behavioral prediction at final layer (per pooling method)
    # ================================================================
    # cosines_per_method and ranks_per_method now hold final-layer values.

    predicted_words = {}
    correct_flags   = {}
    for pm in POOLING_METHODS:
        valid_scores = {
            w: cosines_per_method[pm][w]
            for w in candidates
            if not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        }
        pw = max(valid_scores, key=valid_scores.get) if valid_scores else None
        predicted_words[pm] = pw
        correct_flags[pm]   = (pw in targets) if pw else False

    # Rank aggregation metrics
    rank_metrics = {}
    for pm in POOLING_METHODS:
        target_ranks = [
            ranks_per_method[pm][w]
            for w in candidates
            if cand_meta[w]["word_type"] == "target"
            and not np.isnan(ranks_per_method[pm].get(w, float("nan")))
        ]
        if target_ranks:
            rank_metrics[f"mean_target_rank_{pm}"] = float(np.mean(target_ranks))
            rank_metrics[f"min_target_rank_{pm}"]  = float(np.min(target_ranks))
            rank_metrics[f"max_target_rank_{pm}"]  = float(np.max(target_ranks))
            rank_metrics[f"mrr_{pm}"]              = float(1.0 / np.min(target_ranks))
            rank_metrics[f"hit_at_1_{pm}"]         = float(np.min(target_ranks) == 1)
            rank_metrics[f"hit_at_3_{pm}"]         = float(np.min(target_ranks) <= 3)
            rank_metrics[f"hit_at_5_{pm}"]         = float(np.min(target_ranks) <= 5)
        else:
            for suffix in ["mean_target_rank", "min_target_rank",
                           "max_target_rank", "mrr",
                           "hit_at_1", "hit_at_3", "hit_at_5"]:
                rank_metrics[f"{suffix}_{pm}"] = float("nan")

    # Distance metrics
    distance_metrics = {}
    for pm in POOLING_METHODS:
        tgt_cos = [
            cosines_per_method[pm][w] for w in candidates
            if cand_meta[w]["word_type"] == "target"
            and not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        ]
        non_cos = [
            cosines_per_method[pm][w] for w in candidates
            if cand_meta[w]["word_type"] in ("black", "tan")
            and not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        ]
        distance_metrics[f"mean_cos_hint_targets_{pm}"]    = float(np.mean(tgt_cos)) if tgt_cos else float("nan")
        distance_metrics[f"mean_cos_hint_nontargets_{pm}"] = float(np.mean(non_cos)) if non_cos else float("nan")
        distance_metrics[f"raw_margin_{pm}"] = (
            (float(np.mean(tgt_cos)) - float(np.mean(non_cos)))
            if tgt_cos and non_cos else float("nan")
        )
        valid_scores = {
            w: cosines_per_method[pm][w] for w in candidates
            if not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        }
        sorted_scores = sorted(valid_scores.values(), reverse=True)
        distance_metrics[f"cos_gap_r1_r2_{pm}"] = (
            sorted_scores[0] - sorted_scores[1]
            if len(sorted_scores) >= 2 else float("nan")
        )

    missing_spans = [c for c in candidates if f"cand:{c}" not in spans]

    # --- Memory cleanup ---
    del hidden_states, outputs, layer_hs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    general_record = {
        "row_id"             : row_id,
        "hint"               : hint,
        "n_targets"          : len(targets),
        "n_candidates"       : len(candidates),
        "n_missing_spans"    : len(missing_spans),
        "missing_span_words" : missing_spans,
        "prompt_token_count" : prompt_token_count,
        "use_social_context" : use_social_context,
        "permutation_id"     : permutation_id,
        "giver_features"     : giver_features if use_social_context else {},
    }
    for pm in POOLING_METHODS:
        general_record[f"predicted_word_{pm}"] = predicted_words[pm]
        general_record[f"correct_{pm}"]        = correct_flags[pm]
    general_record.update(rank_metrics)
    general_record.update(distance_metrics)

    return general_record, metrics_records, vector_records


## Cell 8 — Pre-Flight Diagnostic (random-init stability check)

This cell is exclusive to the random-init variant. It runs a forward pass
on the first 5 boards (no_social condition) and verifies that the random
network produces numerically stable hidden states before committing to the
full N=2000 run. Three checks are performed:

1. **NaN / Inf detection** — counts non-finite values in the hidden states
   across all layers. Any non-zero count is a hard stop: fp16 has likely
   overflowed in the residual stream. Fallback: re-instantiate the model
   with `torch.bfloat16` and re-run this cell.

2. **Hidden state norm growth** — reports the mean L2 norm of token hidden
   states at each layer. Random transformers can exhibit exponential norm
   growth across the residual stream. Threshold: if any layer's mean norm
   exceeds 100 (an order of magnitude above what trained Qwen produces),
   flag as a soft warning; the metrics will still be computable, but
   anisotropy interpretation may be affected.

3. **Anisotropy at layer 0** — Random BERT showed L0 anisotropy of 0.357
   in the existing baseline, far higher than trained models (~0.04–0.06)
   because random embedding vectors share more directional bias than
   trained ones. Random Qwen at L0 should also be elevated; we report it
   without setting a hard threshold, but values close to 1.0 across all
   layers would indicate degeneracy.

If all hard checks pass, proceed to Cell 9 (main extraction loop). If any
hard check fails, do not run the main loop.


In [ ]:
print("PRE-FLIGHT DIAGNOSTIC")
print("=" * 60)
print("Running forward pass on 5 boards (no_social condition)...")

_diagnostic_norms_per_layer = [[] for _ in range(NUM_LAYERS + 1)]
_diagnostic_nan_count       = 0
_diagnostic_inf_count       = 0
_diagnostic_aniso_l0        = []

_n_diag = min(5, len(df))
for _i in range(_n_diag):
    _row = df.iloc[_i]
    _prompt, _ = build_prompt(
        hint=str(_row["output"]),
        candidates=list(_row["candidates"]),
        giver_features={},
        use_social_context=False,
    )
    _inputs = tokenizer(_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        _out = model(
            input_ids=_inputs["input_ids"],
            attention_mask=_inputs["attention_mask"],
            output_hidden_states=True,
            return_dict=True,
        )
    for _l in range(NUM_LAYERS + 1):
        _hs = _out.hidden_states[_l][0].detach().float()
        _diagnostic_nan_count += int(torch.isnan(_hs).sum().item())
        _diagnostic_inf_count += int(torch.isinf(_hs).sum().item())
        # Mean L2 norm over tokens
        _norms = torch.norm(_hs, p=2, dim=1)
        _diagnostic_norms_per_layer[_l].append(float(_norms.mean().item()))

    # Anisotropy at L0: mean pairwise cosine across all token positions
    _hs0 = _out.hidden_states[0][0].detach().float()
    if _hs0.shape[0] >= 2:
        _hs0n = _hs0 / (_hs0.norm(p=2, dim=1, keepdim=True) + 1e-8)
        _sim  = _hs0n @ _hs0n.T
        _mask = ~torch.eye(_sim.shape[0], dtype=torch.bool, device=_sim.device)
        _diagnostic_aniso_l0.append(float(_sim[_mask].mean().item()))

    del _out
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# --- Check 1: NaN / Inf ---
print(f"\nCheck 1: NaN / Inf in hidden states")
print(f"  NaN count: {_diagnostic_nan_count}")
print(f"  Inf count: {_diagnostic_inf_count}")
_check1_pass = (_diagnostic_nan_count == 0 and _diagnostic_inf_count == 0)
print(f"  Status: {'PASS' if _check1_pass else 'FAIL — HARD STOP'}")

# --- Check 2: Norm growth ---
print(f"\nCheck 2: Mean hidden-state L2 norm per layer (across {_n_diag} boards)")
print(f"  {'Layer':>6}  {'Mean norm':>12}")
print(f"  {'-'*22}")
_norm_max = 0.0
for _l, _vals in enumerate(_diagnostic_norms_per_layer):
    _m = float(np.mean(_vals))
    _norm_max = max(_norm_max, _m)
    print(f"  {_l:>6}  {_m:>12.4f}")
print(f"  Max mean norm across layers: {_norm_max:.4f}")
_check2_pass = (_norm_max < 100.0)
print(f"  Status: {'PASS' if _check2_pass else 'WARN — norms unusually high'}")

# --- Check 3: Anisotropy at L0 ---
_aniso_l0_mean = float(np.mean(_diagnostic_aniso_l0)) if _diagnostic_aniso_l0 else float("nan")
print(f"\nCheck 3: Mean pairwise cosine (anisotropy) at layer 0")
print(f"  Mean L0 anisotropy: {_aniso_l0_mean:.4f}")
print(f"  Reference: Random BERT L0 anisotropy = 0.357 (interpretable, non-degenerate)")
print(f"  Status: report-only (no hard threshold)")

# --- Summary ---
print(f"\n{'='*60}")
print(f"PRE-FLIGHT SUMMARY")
print(f"{'='*60}")
if _check1_pass and _check2_pass:
    print("  ALL HARD CHECKS PASSED. Safe to run main extraction loop.")
elif not _check1_pass:
    print("  HARD STOP: NaN/Inf detected. Re-instantiate model with bfloat16:")
    print('    model = AutoModelForCausalLM.from_config(config, torch_dtype=torch.bfloat16)')
    print("  Then re-run this diagnostic.")
else:
    print("  WARN: Norms exceed soft threshold (100). Inspect per-layer norm")
    print("  growth above. Main loop may still run; anisotropy should be")
    print("  interpreted carefully in the Results chapter.")


PRE-FLIGHT DIAGNOSTIC
Running forward pass on 5 boards (no_social condition)...

Check 1: NaN / Inf in hidden states
  NaN count: 0
  Inf count: 0
  Status: PASS

Check 2: Mean hidden-state L2 norm per layer (across 5 boards)
   Layer     Mean norm
  ----------------------
       0        1.1938
       1      148.9736
       2      211.6832
       3      259.9986
       4      299.9044
       5      335.2639
       6      367.4310
       7      396.6579
       8      423.7630
       9      449.2611
      10      473.4203
      11      496.5334
      12      518.7084
      13      540.0574
      14      560.7307
      15      580.1819
      16      599.1132
      17      617.3834
      18      634.8500
      19      652.5646
      20      669.4429
      21      685.6746
      22      701.6596
      23      717.3236
      24      732.5520
      25      747.3330
      26      762.2005
      27      776.3919
      28       59.8665
  Max mean norm across layers: 776.3919
  Status: WARN — no

## Cell 9 — Main Extraction Loop

Byte-identical to the trained-Qwen main loop. The `HAS_GENERATION` flag
evaluates to `False` because `generate_response` is not defined in this
notebook, which automatically skips the generation branch (matching the
encoder-only notebooks).


In [ ]:
# --- Sample ---
df_sample = df.sample(
    n=min(SAMPLE_SIZE, len(df)),
    random_state=RANDOM_SEED,
).copy().reset_index(drop=True)

print(f"Sample size: {len(df_sample)} boards")
print(f"Row IDs (first 10): {sorted(df_sample['row_id'].tolist())[:10]} ...")

# --- Pre-select subsample board IDs ---
subsample_size = min(VECTOR_SUBSAMPLE_SIZE, len(df_sample))
subsample_df   = df_sample.sample(n=subsample_size, random_state=RANDOM_SEED)
SUBSAMPLE_IDS  = set(subsample_df["row_id"].tolist())
print(f"Vector subsample: {len(SUBSAMPLE_IDS)} boards")

# --- Pre-generate shuffle seeds for reproducibility ---
rng_shuffles  = np.random.RandomState(RANDOM_SEED + 1000)
shuffle_seeds = rng_shuffles.randint(0, 2**31, size=(len(df_sample), N_SHUFFLES))

os.makedirs(BASE_DIR, exist_ok=True)

# Determine if this notebook is causal (has generate_response) or encoder-only
HAS_GENERATION = "generate_response" in dir()

results = {}

for mode_flag in EXPERIMENT_MODES:
    mode_name = "with_social" if mode_flag else "no_social"
    print(f"\n{'='*60}")
    print(f"Running condition: {mode_name}  "
          f"(canonical + {N_SHUFFLES} shuffles per board)")
    print(f"{'='*60}")

    general_records    = []
    metrics_buffer     = []
    vector_records_all = []
    generation_records = []
    error_log          = []

    # --- Stream A shard tracking (inline; no helper function) ---
    shard_idx       = 0
    boards_in_shard = 0
    shard_paths     = []

    for board_idx, (_, row) in enumerate(
        tqdm(df_sample.iterrows(), total=len(df_sample), desc=mode_name)
    ):
        row_id = int(row["row_id"])
        canonical_candidates = list(row["candidates"])  # alphabetical

        # ==============================================================
        # Canonical ordering (permutation_id = 0)
        # ==============================================================
        try:
            save_vecs = (row_id in SUBSAMPLE_IDS)

            g, m, v = run_instance(
                row=row,
                giver_cols=GIVER_COLS,
                use_social_context=mode_flag,
                candidates_order=canonical_candidates,
                permutation_id=0,
                save_vectors=save_vecs,
            )
            general_records.append(g)
            metrics_buffer.extend(m)
            if v is not None:
                vector_records_all.extend(v)

            # --- Generation (canonical ordering only, causal-only) ---
            if HAS_GENERATION:
                prompt_for_gen, _ = build_prompt(
                    hint=str(row["output"]),
                    candidates=canonical_candidates,
                    giver_features=(
                        extract_giver_features(row, GIVER_COLS)
                        if mode_flag else {}
                    ),
                    use_social_context=mode_flag,
                )
                gen_result = generate_response(
                    prompt=prompt_for_gen,
                    candidates=canonical_candidates,
                    max_new_tokens=GENERATION_MAX_TOKENS,
                )
                gen_record = {
                    "row_id"                  : row_id,
                    "use_social_context"      : mode_flag,
                    "generated_text"          : gen_result["generated_text"],
                    "generated_word"          : gen_result["generated_word"],
                    "generated_in_candidates" : gen_result["generated_in_candidates"],
                    "generated_correct"       : (
                        gen_result["generated_word"] in set(row["targets"])
                        if gen_result["generated_word"] else False
                    ),
                }
                for pm in POOLING_METHODS:
                    gen_record[f"concordance_{pm}"] = (
                        gen_result["generated_word"] == g[f"predicted_word_{pm}"]
                        if gen_result["generated_word"] else False
                    )
                generation_records.append(gen_record)

        except Exception as e:
            error_log.append({"row_id": row_id, "error": str(e), "permutation_id": 0})
            print(f"  ERROR row_id={row_id} perm=0: {e}")

        # ==============================================================
        # Shuffle permutations (permutation_id = 1..K)
        # ==============================================================
        for k in range(N_SHUFFLES):
            try:
                perm_rng = np.random.RandomState(int(shuffle_seeds[board_idx, k]))
                shuffled_candidates = list(canonical_candidates)
                perm_rng.shuffle(shuffled_candidates)

                g_shuf, m_shuf, _ = run_instance(
                    row=row,
                    giver_cols=GIVER_COLS,
                    use_social_context=mode_flag,
                    candidates_order=shuffled_candidates,
                    permutation_id=k + 1,
                    save_vectors=False,
                )
                general_records.append(g_shuf)
                metrics_buffer.extend(m_shuf)
            except Exception as e:
                error_log.append({"row_id": row_id, "error": str(e), "permutation_id": k + 1})
                print(f"  ERROR row_id={row_id} perm={k+1}: {e}")

        # --- Shard flush check (INLINE) ---
        boards_in_shard += 1
        if boards_in_shard >= SHARD_BOARDS and metrics_buffer:
            shard_path = os.path.join(
                BASE_DIR,
                f"{MODEL_PREFIX}_metrics_{mode_name}_shard{shard_idx:03d}.parquet",
            )
            pd.DataFrame(metrics_buffer).to_parquet(shard_path, index=False)
            shard_paths.append(shard_path)
            metrics_buffer = []
            shard_idx += 1
            boards_in_shard = 0
            gc.collect()

    # --- Final flush of remaining buffer (INLINE) ---
    if metrics_buffer:
        shard_path = os.path.join(
            BASE_DIR,
            f"{MODEL_PREFIX}_metrics_{mode_name}_shard{shard_idx:03d}.parquet",
        )
        pd.DataFrame(metrics_buffer).to_parquet(shard_path, index=False)
        shard_paths.append(shard_path)
        metrics_buffer = []
        shard_idx += 1
        gc.collect()

    # ------------------------------------------------------------------
    # Concatenate shards into a single parquet file
    # ------------------------------------------------------------------
    metrics_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_metrics_{mode_name}.parquet")
    if shard_paths:
        all_shards = [pd.read_parquet(p) for p in shard_paths]
        metrics_df = pd.concat(all_shards, ignore_index=True)
        metrics_df.to_parquet(metrics_path, index=False)
        for p in shard_paths:
            os.remove(p)
        del all_shards
        gc.collect()
        metrics_mb = os.path.getsize(metrics_path) / 1e6
    else:
        metrics_df = pd.DataFrame()
        metrics_mb = 0.0

    # ------------------------------------------------------------------
    # Save Stream B: Vector subsample
    # ------------------------------------------------------------------
    n_vec_records = len(vector_records_all)
    vec_mb = 0.0
    vec_index_df = pd.DataFrame()
    if n_vec_records > 0:
        vec_index_rows = []
        vec_arrays     = []
        for i, rec in enumerate(vector_records_all):
            vec = rec["vector"]
            valid = (vec is not None and isinstance(vec, np.ndarray)
                     and vec.shape == (HIDDEN_DIM,))
            vec_arrays.append(vec if valid else None)
            vec_index_rows.append({
                "record_idx"        : i,
                "row_id"            : rec["row_id"],
                "layer"             : rec["layer"],
                "word"              : rec["word"],
                "word_type"         : rec["word_type"],
                "token_count"       : rec["token_count"],
                "pooling_method"    : rec["pooling_method"],
                "use_social_context": rec["use_social_context"],
                "vector_valid"      : valid,
            })
        vec_index_df = pd.DataFrame(vec_index_rows)

        valid_mask = np.array([v is not None for v in vec_arrays])
        valid_idx  = np.where(valid_mask)[0]
        matrix     = np.zeros((n_vec_records, HIDDEN_DIM), dtype=np.float16)
        if len(valid_idx):
            matrix[valid_idx] = np.stack([vec_arrays[i] for i in valid_idx])

        del vec_arrays
        gc.collect()

        vec_index_path = os.path.join(
            BASE_DIR, f"{MODEL_PREFIX}_vectors_subsample_index_{mode_name}.csv"
        )
        vec_index_df.to_csv(vec_index_path, index=False)

        vec_matrix_path = os.path.join(
            BASE_DIR, f"{MODEL_PREFIX}_vectors_subsample_{mode_name}_f16.npz"
        )
        np.savez_compressed(vec_matrix_path, vectors=matrix)

        # NPZ integrity check
        _v = np.load(vec_matrix_path)
        _shape = _v["vectors"].shape
        _v.close()
        vec_mb = os.path.getsize(vec_matrix_path) / 1e6
        print(f"  Subsample NPZ verified: shape={_shape}, {vec_mb:.1f} MB")

        del matrix

    # ------------------------------------------------------------------
    # Save General + Generation
    # ------------------------------------------------------------------
    general_df = pd.DataFrame(general_records)
    general_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_general_{mode_name}.csv")
    general_df.to_csv(general_path, index=False)

    generation_df = pd.DataFrame(generation_records)
    if HAS_GENERATION and len(generation_df) > 0:
        generation_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_generation_{mode_name}.csv")
        generation_df.to_csv(generation_path, index=False)

    print(f"\nCondition '{mode_name}' complete.")
    print(f"  Boards processed     : {len(df_sample)}")
    print(f"  Permutations/board   : 1 canonical + {N_SHUFFLES} shuffles")
    print(f"  General records      : {len(general_df):,}")
    print(f"  Metrics rows         : {len(metrics_df):,}  ({metrics_mb:.1f} MB)")
    print(f"  Subsample vectors    : {n_vec_records:,} records  ({vec_mb:.1f} MB)")
    print(f"  Generation rows      : {len(generation_df)}")
    print(f"  Errors               : {len(error_log)}")

    results[mode_name] = {
        "general_df"    : general_df,
        "metrics_df"    : metrics_df,
        "generation_df" : generation_df,
        "error_log"     : error_log,
    }

    del vector_records_all, general_records, generation_records
    vector_records_all = []  # reset for next condition
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"\nBoth conditions complete. Outputs in: {BASE_DIR}")


Sample size: 2000 boards
Row IDs (first 10): [0, 8, 14, 15, 17, 19, 23, 31, 37, 41] ...
Vector subsample: 100 boards

Running condition: no_social  (canonical + 2 shuffles per board)


no_social:   0%|          | 0/2000 [00:00<?, ?it/s]

  Subsample NPZ verified: shape=(104110, 3584), 367.5 MB

Condition 'no_social' complete.
  Boards processed     : 2000
  Permutations/board   : 1 canonical + 2 shuffles
  General records      : 6,000
  Metrics rows         : 3,123,039  (48.9 MB)
  Subsample vectors    : 104,110 records  (367.5 MB)
  Generation rows      : 0
  Errors               : 0

Running condition: with_social  (canonical + 2 shuffles per board)


with_social:   0%|          | 0/2000 [00:00<?, ?it/s]

  Subsample NPZ verified: shape=(147378, 3584), 654.7 MB

Condition 'with_social' complete.
  Boards processed     : 2000
  Permutations/board   : 1 canonical + 2 shuffles
  General records      : 6,000
  Metrics rows         : 4,399,416  (58.4 MB)
  Subsample vectors    : 147,378 records  (654.7 MB)
  Generation rows      : 0
  Errors               : 0

Both conditions complete. Outputs in: /content/drive/MyDrive/TCC/random_qwen_outputs


## SC1 — Prompt Structure Verification


In [ ]:
print("SC1: Prompt Structure Verification")
print("=" * 60)

test_row = df_sample.iloc[0]

for mode_flag in [False, True]:
    mode_name = "with_social" if mode_flag else "no_social"
    giver_features = (
        extract_giver_features(test_row, GIVER_COLS)
        if mode_flag else {}
    )
    prompt, feature_markers = build_prompt(
        hint=test_row["output"],
        candidates=list(test_row["candidates"]),
        giver_features=giver_features,
        use_social_context=mode_flag,
    )

    hint_found       = test_row["output"] in prompt
    all_cands_found  = all(c in prompt for c in test_row["candidates"])
    if mode_flag:
        markers_found = all(m in prompt for m in feature_markers.values())
        markers_leaked = False
    else:
        markers_found  = True
        markers_leaked = any(
            f"{label}:" in prompt for label in _FEATURE_LABEL_MAP.values()
        )

    token_count = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])

    print(f"\nCondition: {mode_name}")
    print(f"  Hint found in prompt       : {hint_found}")
    print(f"  All candidates found       : {all_cands_found}")
    if mode_flag:
        print(f"  All giver markers found    : {markers_found}")
    else:
        print(f"  Giver markers leaked       : {markers_leaked}  (must be False)")
    print(f"  Prompt token count         : {token_count}")
    print(f"  Prompt preview:")
    preview = prompt[:300].replace(chr(10), " | ")
    print(f"    {preview}")


SC1: Prompt Structure Verification

Condition: no_social
  Hint found in prompt       : True
  All candidates found       : True
  Giver markers leaked       : False  (must be False)
  Prompt token count         : 132
  Prompt preview:
    <|im_start|>system | You are a helpful assistant.<|im_end|> | <|im_start|>user | You are playing the game Codenames. | The hint is: "event" | The possible words are: | 1. capital | 2. cold | 3. cover | 4. date | 5. day | 6. degree | 7. field | 8. grace | 9. march | 10. novel | 11. opera | 12. part | 13. soul | 14. spell | 15. spring | 16. t

Condition: with_social
  Hint found in prompt       : True
  All candidates found       : True
  All giver markers found    : True
  Prompt token count         : 187
  Prompt preview:
    <|im_start|>system | You are a helpful assistant.<|im_end|> | <|im_start|>user | You are playing the game Codenames. | The clue was given by a player with the following characteristics: | Marriage: single, Education: bachelor, R

## SC2 — Span Coverage and Token Count Statistics


In [ ]:
print("SC2: Span Coverage and Token Count Statistics")
print("=" * 60)

for mode_name in ["no_social", "with_social"]:
    mdf = results[mode_name]["metrics_df"]
    if len(mdf) == 0:
        print(f"\n{mode_name}: no metrics rows; skipping.")
        continue
    mdf_canon = mdf[mdf["permutation_id"] == 0]
    mdf_l0    = mdf_canon[mdf_canon["layer"] == 0]

    cand_rows = mdf_l0[mdf_l0["word_type"].isin(["target", "black", "tan"])]
    total_cand_slots   = len(cand_rows)
    missing_cand_slots = int(cand_rows["cosine_to_hint_mean"].isna().sum())
    coverage_pct = 100.0 * (1 - missing_cand_slots / max(total_cand_slots, 1))

    print(f"\nCondition: {mode_name}")
    print(f"  Candidate slots   : {total_cand_slots}")
    print(f"  Missing spans     : {missing_cand_slots}")
    print(f"  Span coverage     : {coverage_pct:.2f}%")
    if coverage_pct < 95.0:
        print("  WARNING: Span coverage below 95%. Review span detection logic.")

    print(f"\n  Token count per word type at layer 0:")
    for wt in ["hint", "target", "black", "tan", "giver_feature"]:
        subset = mdf_l0[mdf_l0["word_type"] == wt]["token_count"]
        if len(subset) > 0:
            print(f"    {wt:14s}: mean={subset.mean():.2f}, "
                  f"std={subset.std():.2f}, max={int(subset.max())}")


SC2: Span Coverage and Token Count Statistics

Condition: no_social
  Candidate slots   : 33897
  Missing spans     : 0
  Span coverage     : 100.00%

  Token count per word type at layer 0:
    hint          : mean=1.44, std=0.64, max=7
    target        : mean=1.06, std=0.33, max=4
    black         : mean=1.06, std=0.33, max=4
    tan           : mean=1.06, std=0.34, max=4

Condition: with_social
  Candidate slots   : 33897
  Missing spans     : 0
  Span coverage     : 100.00%

  Token count per word type at layer 0:
    hint          : mean=1.44, std=0.64, max=7
    target        : mean=1.06, std=0.33, max=4
    black         : mean=1.06, std=0.33, max=4
    tan           : mean=1.06, std=0.34, max=4
    giver_feature : mean=3.81, std=1.27, max=8


## SC3 — Anisotropy Characterization


In [ ]:
print("SC3: Anisotropy Characterization (mean pooling, no_social)")
print("=" * 60)

mdf = results["no_social"]["metrics_df"]
if len(mdf) > 0:
    mdf_canon = mdf[mdf["permutation_id"] == 0]
    aniso_per_layer = (
        mdf_canon.groupby("layer")
        .agg(
            mean_aniso=("layer_mean_pairwise_cosine", "mean"),
            std_aniso =("layer_std_pairwise_cosine",  "mean"),
            n_boards  =("row_id", "nunique"),
        )
        .reset_index()
    )
    print(f"\n{'Layer':>6}  {'Mean aniso':>12}  {'Std aniso':>12}  {'N boards':>10}")
    print("-" * 50)
    for _, r in aniso_per_layer.iterrows():
        print(f"{int(r['layer']):>6}  {r['mean_aniso']:>12.4f}  "
              f"{r['std_aniso']:>12.4f}  {int(r['n_boards']):>10}")


SC3: Anisotropy Characterization (mean pooling, no_social)

 Layer    Mean aniso     Std aniso    N boards
--------------------------------------------------
     0        0.0002        0.0179        2000
     1        0.1243        0.0448        2000
     2        0.1020        0.0360        2000
     3        0.0908        0.0318        2000
     4        0.0803        0.0291        2000
     5        0.0745        0.0275        2000
     6        0.0714        0.0264        2000
     7        0.0668        0.0256        2000
     8        0.0618        0.0249        2000
     9        0.0589        0.0243        2000
    10        0.0570        0.0239        2000
    11        0.0551        0.0235        2000
    12        0.0535        0.0232        2000
    13        0.0522        0.0229        2000
    14        0.0512        0.0227        2000
    15        0.0503        0.0225        2000
    16        0.0492        0.0222        2000
    17        0.0480        0.0221        2

## SC4 — Behavioral Accuracy Summary

`HAS_GENERATION` evaluates to False here, so only the cosine-rank
accuracy block is reported. The generation-based accuracy and concordance
sections are silently skipped.


In [ ]:
def wilson_confidence_interval(successes: int, n: int, z: float = 1.96):
    """Wilson score interval for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    p_hat  = successes / n
    denom  = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


print("SC4: Behavioral Accuracy Summary")
print("=" * 60)

for mode_name in ["no_social", "with_social"]:
    gdf       = results[mode_name]["general_df"]
    gdf_canon = gdf[gdf["permutation_id"] == 0]
    gen_df    = results[mode_name]["generation_df"]
    n         = len(gdf_canon)

    print(f"\nCondition: {mode_name}  (N={n})")

    for pm in POOLING_METHODS:
        n_correct = int(gdf_canon[f"correct_{pm}"].sum())
        accuracy  = n_correct / n if n > 0 else float("nan")
        ci_lo, ci_hi = wilson_confidence_interval(n_correct, n)

        mrr  = gdf_canon[f"mrr_{pm}"].dropna().mean()
        hit1 = gdf_canon[f"hit_at_1_{pm}"].dropna().mean()
        hit3 = gdf_canon[f"hit_at_3_{pm}"].dropna().mean()
        hit5 = gdf_canon[f"hit_at_5_{pm}"].dropna().mean()
        mean_rank = gdf_canon[f"mean_target_rank_{pm}"].dropna().mean()
        std_rank  = gdf_canon[f"mean_target_rank_{pm}"].dropna().std()

        mean_n_cands   = float(gdf_canon["n_candidates"].mean())
        mean_n_targets = float(gdf_canon["n_targets"].mean())
        random_rank    = (mean_n_cands + 1) / (mean_n_targets + 1)

        print(f"\n  Cosine-rank ({pm}):")
        print(f"    Top-1 accuracy    : {accuracy:.3f} ({n_correct}/{n})")
        print(f"    Wilson 95% CI     : [{ci_lo:.3f}, {ci_hi:.3f}]")
        print(f"    MRR               : {mrr:.4f}")
        print(f"    Hit@1 / @3 / @5   : {hit1:.3f} / {hit3:.3f} / {hit5:.3f}")
        print(f"    Mean target rank  : {mean_rank:.2f} ± {std_rank:.2f}")
        print(f"    Random baseline   : {random_rank:.2f}")

    if HAS_GENERATION and len(gen_df) > 0:
        n_gen        = len(gen_df)
        gen_in_cands = int(gen_df["generated_in_candidates"].sum())
        gen_correct  = int(gen_df["generated_correct"].sum())
        gen_acc      = gen_correct / n_gen if n_gen > 0 else float("nan")
        gen_ci       = wilson_confidence_interval(gen_correct, n_gen)

        print(f"\n  Generation-based accuracy:")
        print(f"    Generated in candidates : {gen_in_cands}/{n_gen} "
              f"({gen_in_cands/n_gen:.3f})")
        print(f"    Generation accuracy     : {gen_acc:.3f} ({gen_correct}/{n_gen})")
        print(f"    Wilson 95% CI           : [{gen_ci[0]:.3f}, {gen_ci[1]:.3f}]")

        for pm in POOLING_METHODS:
            concordance = gen_df[f"concordance_{pm}"].mean()
            print(f"    Concordance with {pm:8s}: {concordance:.3f}")


SC4: Behavioral Accuracy Summary

Condition: no_social  (N=2000)

  Cosine-rank (mean):
    Top-1 accuracy    : 0.079 (158/2000)
    Wilson 95% CI     : [0.068, 0.092]
    MRR               : 0.2282
    Hit@1 / @3 / @5   : 0.079 / 0.204 / 0.331
    Mean target rank  : 8.99 ± 4.59
    Random baseline   : 8.08

  Cosine-rank (max_norm):
    Top-1 accuracy    : 0.076 (153/2000)
    Wilson 95% CI     : [0.066, 0.089]
    MRR               : 0.2268
    Hit@1 / @3 / @5   : 0.076 / 0.210 / 0.339
    Mean target rank  : 9.05 ± 4.64
    Random baseline   : 8.08

Condition: with_social  (N=2000)

  Cosine-rank (mean):
    Top-1 accuracy    : 0.059 (119/2000)
    Wilson 95% CI     : [0.050, 0.071]
    MRR               : 0.2125
    Hit@1 / @3 / @5   : 0.059 / 0.199 / 0.328
    Mean target rank  : 9.17 ± 4.63
    Random baseline   : 8.08

  Cosine-rank (max_norm):
    Top-1 accuracy    : 0.062 (125/2000)
    Wilson 95% CI     : [0.053, 0.074]
    MRR               : 0.2137
    Hit@1 / @3 / @5   : 

## SC5 — Layer-wise Margin Curve (Anisotropy-Adjusted)


In [ ]:
print("SC5: Layer-wise Margin Curve")
print("=" * 60)

for pm in POOLING_METHODS:
    print(f"\n--- Pooling: {pm} ---")

    for mode_name in ["no_social", "with_social"]:
        mdf = results[mode_name]["metrics_df"]
        if len(mdf) == 0:
            continue
        mdf_canon = mdf[mdf["permutation_id"] == 0]

        layer_margin_records = []

        for layer_idx in range(NUM_LAYERS + 1):
            layer_data = mdf_canon[mdf_canon["layer"] == layer_idx]

            board_margins  = []
            board_hint_tgt = []
            board_hint_non = []

            for row_id, board_df in layer_data.groupby("row_id"):
                tgt_cos = board_df[board_df["word_type"] == "target"][f"cosine_to_hint_{pm}"].dropna()
                non_cos = board_df[board_df["word_type"].isin(["black", "tan"])][f"cosine_to_hint_{pm}"].dropna()
                if len(tgt_cos) == 0 or len(non_cos) == 0:
                    continue
                mt = float(tgt_cos.mean())
                mn = float(non_cos.mean())
                board_margins.append(mt - mn)
                board_hint_tgt.append(mt)
                board_hint_non.append(mn)

            if board_margins:
                aniso_vals     = layer_data.groupby("row_id")["layer_mean_pairwise_cosine"].first().dropna()
                aniso_std_vals = layer_data.groupby("row_id")["layer_std_pairwise_cosine"].first().dropna()
                layer_mean_aniso = float(aniso_vals.mean()) if len(aniso_vals) > 0 else float("nan")
                layer_std_aniso  = float(aniso_std_vals.mean()) if len(aniso_std_vals) > 0 else float("nan")

                raw_margin = float(np.mean(board_margins))
                adjusted_margin = (
                    raw_margin / layer_std_aniso
                    if not np.isnan(layer_std_aniso) and layer_std_aniso > 0
                    else float("nan")
                )

                layer_margin_records.append({
                    "layer"           : layer_idx,
                    "pooling_method"  : pm,
                    "condition"       : mode_name,
                    "mean_margin"     : raw_margin,
                    "std_margin"      : float(np.std(board_margins)),
                    "adjusted_margin" : adjusted_margin,
                    "mean_hint_tgt"   : float(np.mean(board_hint_tgt)),
                    "mean_hint_non"   : float(np.mean(board_hint_non)),
                    "mean_anisotropy" : layer_mean_aniso,
                    "n_boards"        : len(board_margins),
                })

        margin_df = pd.DataFrame(layer_margin_records)

        print(f"\n  Condition: {mode_name}")
        print(f"  {'Layer':>6}  {'Raw':>8}  {'Adj':>8}  {'hint-tgt':>10}  {'hint-non':>10}  {'Aniso':>8}")
        print(f"  {'-'*60}")
        for _, r in margin_df.iterrows():
            print(f"  {int(r['layer']):>6}  {r['mean_margin']:>8.5f}  "
                  f"{r['adjusted_margin']:>8.4f}  "
                  f"{r['mean_hint_tgt']:>10.5f}  "
                  f"{r['mean_hint_non']:>10.5f}  "
                  f"{r['mean_anisotropy']:>8.5f}")

        if len(margin_df) > 0:
            best_raw = margin_df.loc[margin_df["mean_margin"].idxmax()]
            best_adj = margin_df.loc[margin_df["adjusted_margin"].idxmax()]
            print(f"\n  Peak raw: layer {int(best_raw['layer'])} ({best_raw['mean_margin']:.5f})")
            print(f"  Peak adj: layer {int(best_adj['layer'])} ({best_adj['adjusted_margin']:.4f})")

        margin_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_layer_margins_{pm}_{mode_name}.csv")
        margin_df.to_csv(margin_path, index=False)


SC5: Layer-wise Margin Curve

--- Pooling: mean ---

  Condition: no_social
   Layer       Raw       Adj    hint-tgt    hint-non     Aniso
  ------------------------------------------------------------
       0  -0.00057   -0.0319    -0.00033     0.00024   0.00021
       1   0.00079    0.0176     0.08929     0.08850   0.12423
       2  -0.00011   -0.0032     0.07691     0.07703   0.10195
       3   0.00013    0.0040     0.07175     0.07162   0.09074
       4   0.00013    0.0043     0.06478     0.06465   0.08031
       5  -0.00022   -0.0079     0.06093     0.06115   0.07446
       6  -0.00022   -0.0083     0.05982     0.06004   0.07140
       7  -0.00005   -0.0021     0.05701     0.05706   0.06680
       8  -0.00020   -0.0081     0.05312     0.05332   0.06179
       9  -0.00057   -0.0234     0.05066     0.05123   0.05890
      10  -0.00028   -0.0117     0.04996     0.05024   0.05695
      11   0.00001    0.0005     0.04938     0.04936   0.05513
      12   0.00015    0.0065     0.04822  

## SC6 — Per-Layer Positional Confound (Spearman ρ at every layer)


In [ ]:
print("SC6: Per-Layer Positional Confound")
print("=" * 60)

# --- Order consistency check (folded from old SC4) ---
gdf_ns = results["no_social"]["general_df"]
gdf_ws = results["with_social"]["general_df"]
gdf_ns_canon = gdf_ns[gdf_ns["permutation_id"] == 0].set_index("row_id")
gdf_ws_canon = gdf_ws[gdf_ws["permutation_id"] == 0].set_index("row_id")
common_rows = set(gdf_ns_canon.index) & set(gdf_ws_canon.index)
print(f"Order consistency: {len(common_rows)} common boards (both conditions saw same alphabetical ordering)")

# --- Per-layer Spearman ρ ---
mdf = results["no_social"]["metrics_df"]
if len(mdf) == 0:
    print("No metrics rows; skipping SC6.")
else:
    mdf_canon = mdf[mdf["permutation_id"] == 0]

    confound_records = []

    for layer_idx in range(NUM_LAYERS + 1):
        layer_data = mdf_canon[
            (mdf_canon["layer"] == layer_idx) &
            (mdf_canon["word_type"].isin(["target", "black", "tan"]))
        ]

        layer_rhos = []
        for row_id, board_df in layer_data.groupby("row_id"):
            positions = board_df["list_position"].values
            cosines   = board_df["cosine_to_hint_mean"].values
            valid = ~np.isnan(cosines) & ~np.isnan(positions)
            if valid.sum() < 3:
                continue
            rho, _ = spearmanr(positions[valid], cosines[valid])
            layer_rhos.append(rho)

        if layer_rhos:
            confound_records.append({
                "layer"    : layer_idx,
                "mean_rho" : float(np.mean(layer_rhos)),
                "std_rho"  : float(np.std(layer_rhos)),
                "n_boards" : len(layer_rhos),
            })

    confound_df = pd.DataFrame(confound_records)

    print(f"\n{'Layer':>6}  {'Mean rho':>10}  {'Std rho':>10}  {'Concern?':>10}")
    print("-" * 45)
    for _, r in confound_df.iterrows():
        concern = "YES" if abs(r["mean_rho"]) > 0.1 else ""
        print(f"{int(r['layer']):>6}  {r['mean_rho']:>10.4f}  {r['std_rho']:>10.4f}  {concern:>10}")

    confound_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_position_confound_by_layer.csv")
    confound_df.to_csv(confound_path, index=False)
    print(f"\nSaved: {confound_path}")

    # --- Final-layer Mann-Whitney U for backward compatibility ---
    final_data = mdf_canon[
        (mdf_canon["layer"] == NUM_LAYERS) &
        (mdf_canon["word_type"].isin(["target", "black", "tan"]))
    ]
    positions_all = final_data["list_position"].values
    cosines_all   = final_data["cosine_to_hint_mean"].values
    valid_mask    = ~np.isnan(cosines_all) & ~np.isnan(positions_all)
    pv, cv = positions_all[valid_mask], cosines_all[valid_mask]

    if len(pv) >= 10:
        rho_f, p_rho = spearmanr(pv, cv)
        med = np.median(pv)
        near_c, far_c = cv[pv <= med], cv[pv > med]
        if len(near_c) >= 2 and len(far_c) >= 2:
            u_stat, u_p = mannwhitneyu(near_c, far_c, alternative="greater")
            eff_r = u_stat / (len(near_c) * len(far_c))
        else:
            u_stat, u_p, eff_r = float("nan"), float("nan"), float("nan")
        print(f"\n  Final-layer Mann-Whitney U:")
        print(f"    Spearman rho = {rho_f:+.4f} (p={p_rho:.4f})")
        print(f"    U={u_stat:.1f}, p={u_p:.4f}, r={eff_r:.4f}")
        print(f"    Near cos={near_c.mean():.5f}, Far cos={far_c.mean():.5f}")


SC6: Per-Layer Positional Confound
Order consistency: 2000 common boards (both conditions saw same alphabetical ordering)

 Layer    Mean rho     Std rho    Concern?
---------------------------------------------
     0     -0.0094      0.2498            
     1     -0.3851      0.2389         YES
     2     -0.3573      0.2414         YES
     3     -0.3478      0.2423         YES
     4     -0.3290      0.2384         YES
     5     -0.3175      0.2414         YES
     6     -0.3091      0.2411         YES
     7     -0.3001      0.2398         YES
     8     -0.3033      0.2384         YES
     9     -0.2971      0.2393         YES
    10     -0.2915      0.2373         YES
    11     -0.2897      0.2360         YES
    12     -0.2815      0.2364         YES
    13     -0.2935      0.2341         YES
    14     -0.2977      0.2351         YES
    15     -0.2824      0.2389         YES
    16     -0.2804      0.2379         YES
    17     -0.2726      0.2416         YES
    18     -0.

## SC7 — Shuffle Confound Decomposition


In [ ]:
if N_SHUFFLES > 0:
    print("SC7: Shuffle Confound Decomposition")
    print("=" * 60)

    mode_name = "no_social"
    mdf = results[mode_name]["metrics_df"]

    if len(mdf) == 0:
        print("No metrics rows; skipping SC7.")
    else:
        # Final layer headline
        final_data = mdf[
            (mdf["layer"] == NUM_LAYERS) &
            (mdf["word_type"].isin(["target", "black", "tan"]))
        ]
        word_var    = final_data.groupby(["row_id", "word"])["cosine_to_hint_mean"].var().dropna()
        between_var = final_data.groupby(["row_id", "permutation_id"])["cosine_to_hint_mean"].var().dropna()
        mean_within  = float(word_var.mean())
        mean_between = float(between_var.mean())
        total = mean_within + mean_between
        ratio = mean_between / total if total > 0 else float("nan")

        print(f"\n  Final layer (layer {NUM_LAYERS}), mean pooling, no_social:")
        print(f"  Mean within-word variance  (positional) : {mean_within:.6f}")
        print(f"  Mean between-word variance (semantic)   : {mean_between:.6f}")
        print(f"  Semantic signal ratio                   : {ratio:.4f}")
        print(f"    (1.0 = pure semantic, 0.0 = pure positional)")

        # Per-layer
        print(f"\n  Per-layer semantic signal ratio:")
        print(f"  {'Layer':>6}  {'Within (pos)':>14}  {'Between (sem)':>14}  {'Ratio':>8}")
        print(f"  {'-'*48}")

        shuffle_records = []
        for layer_idx in range(NUM_LAYERS + 1):
            ld = mdf[
                (mdf["layer"] == layer_idx) &
                (mdf["word_type"].isin(["target", "black", "tan"]))
            ]
            wv = ld.groupby(["row_id", "word"])["cosine_to_hint_mean"].var().dropna()
            bv = ld.groupby(["row_id", "permutation_id"])["cosine_to_hint_mean"].var().dropna()
            mwv = float(wv.mean()) if len(wv) > 0 else 0.0
            mbv = float(bv.mean()) if len(bv) > 0 else 0.0
            tv  = mwv + mbv
            r   = mbv / tv if tv > 0 else float("nan")
            print(f"  {layer_idx:>6}  {mwv:>14.6f}  {mbv:>14.6f}  {r:>8.4f}")
            shuffle_records.append({
                "layer": layer_idx, "within_var": mwv,
                "between_var": mbv, "semantic_ratio": r,
            })

        shuffle_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
        pd.DataFrame(shuffle_records).to_csv(shuffle_path, index=False)
        print(f"\n  Saved: {shuffle_path}")
else:
    print("\nSC7 skipped (N_SHUFFLES = 0).")


SC7: Shuffle Confound Decomposition

  Final layer (layer 28), mean pooling, no_social:
  Mean within-word variance  (positional) : 0.000314
  Mean between-word variance (semantic)   : 0.000388
  Semantic signal ratio                   : 0.5522
    (1.0 = pure semantic, 0.0 = pure positional)

  Per-layer semantic signal ratio:
   Layer    Within (pos)   Between (sem)     Ratio
  ------------------------------------------------
       0        0.000000        0.000289    0.9984
       1        0.000380        0.000933    0.7104
       2        0.000299        0.000638    0.6807
       3        0.000281        0.000544    0.6591
       4        0.000275        0.000496    0.6437
       5        0.000275        0.000467    0.6296
       6        0.000273        0.000447    0.6207
       7        0.000277        0.000436    0.6118
       8        0.000282        0.000428    0.6029
       9        0.000285        0.000421    0.5958
      10        0.000288        0.000417    0.5915
      1

## Save Outputs Summary


In [ ]:
# Save error logs
for mode_name in ["no_social", "with_social"]:
    errors = results[mode_name]["error_log"]
    if errors:
        err_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_errors_{mode_name}.csv")
        pd.DataFrame(errors).to_csv(err_path, index=False)
        print(f"Saved error log: {err_path}  ({len(errors)} errors)")

print("\n" + "=" * 60)
print("OUTPUT SUMMARY")
print("=" * 60)
print(f"Directory: {BASE_DIR}\n")

for mode_name in ["no_social", "with_social"]:
    print(f"  [{mode_name}]")
    candidate_files = [
        f"{MODEL_PREFIX}_general_{mode_name}.csv",
        f"{MODEL_PREFIX}_metrics_{mode_name}.parquet",
        f"{MODEL_PREFIX}_vectors_subsample_index_{mode_name}.csv",
        f"{MODEL_PREFIX}_vectors_subsample_{mode_name}_f16.npz",
    ]
    if HAS_GENERATION:
        candidate_files.append(f"{MODEL_PREFIX}_generation_{mode_name}.csv")
    for suffix in candidate_files:
        fpath = os.path.join(BASE_DIR, suffix)
        if os.path.exists(fpath):
            print(f"    {suffix}  ({os.path.getsize(fpath)/1e6:.1f} MB)")
        else:
            print(f"    {suffix}  [NOT FOUND]")

print(f"\n  Aggregate files:")
print(f"    {MODEL_PREFIX}_position_confound_by_layer.csv")
if N_SHUFFLES > 0:
    print(f"    {MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
for pm in POOLING_METHODS:
    for mn in ["no_social", "with_social"]:
        print(f"    {MODEL_PREFIX}_layer_margins_{pm}_{mn}.csv")

print("\n" + "=" * 60)
print("LOADING PATTERN FOR DOWNSTREAM NOTEBOOKS")
print("=" * 60)
print(f'''
# --- Stream A: Metrics (all boards, all permutations, no vectors) ---
metrics = pd.read_parquet(".../{MODEL_PREFIX}_metrics_no_social.parquet")
metrics_canonical = metrics[metrics["permutation_id"] == 0]

# --- Stream B: Vector subsample (canonical only) ---
index  = pd.read_csv(".../{MODEL_PREFIX}_vectors_subsample_index_no_social.csv")
data   = np.load(".../{MODEL_PREFIX}_vectors_subsample_no_social_f16.npz")
matrix = data["vectors"]   # shape [N, {{HIDDEN_DIM}}], dtype float16
vec    = matrix[i].astype(np.float32)

# Filter by pooling method:
idx_mean = index[index["pooling_method"] == "mean"]
idx_maxn = index[index["pooling_method"] == "max_norm"]

# --- Generation results (causal-only) ---
gen = pd.read_csv(".../{MODEL_PREFIX}_generation_no_social.csv")

# --- Shuffle analysis ---
shuffles = pd.read_csv(".../{MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
''')



OUTPUT SUMMARY
Directory: /content/drive/MyDrive/TCC/random_qwen_outputs

  [no_social]
    random_qwen_general_no_social.csv  (1.8 MB)
    random_qwen_metrics_no_social.parquet  (48.9 MB)
    random_qwen_vectors_subsample_index_no_social.csv  (4.6 MB)
    random_qwen_vectors_subsample_no_social_f16.npz  (367.5 MB)
  [with_social]
    random_qwen_general_with_social.csv  (3.2 MB)
    random_qwen_metrics_with_social.parquet  (58.4 MB)
    random_qwen_vectors_subsample_index_with_social.csv  (7.2 MB)
    random_qwen_vectors_subsample_with_social_f16.npz  (654.7 MB)

  Aggregate files:
    random_qwen_position_confound_by_layer.csv
    random_qwen_shuffle_decomposition_by_layer.csv
    random_qwen_layer_margins_mean_no_social.csv
    random_qwen_layer_margins_mean_with_social.csv
    random_qwen_layer_margins_max_norm_no_social.csv
    random_qwen_layer_margins_max_norm_with_social.csv

LOADING PATTERN FOR DOWNSTREAM NOTEBOOKS

# --- Stream A: Metrics (all boards, all permutations, no ve